# Profiling in PyTorch

This notbook follows the [Profiling in PyTorch](https://huggingface.co/blog/torch-profiler) Hugging Face tutorial.

In [6]:
import os
import torch

## Part 1: A Beginner's Guide to `torch.profiler`

The following code runs and profiles a function `fn` consisting of matrix multiplication and addition.

In [11]:
# Parameters
size = 64
dtype = torch.bfloat16
compile_fn = False
warmup_steps = 3
profile_steps = 5
device = "cuda"
trace_dir = "./temp/01_matmul_add"

# Function to be profiled (matrix multiplication and addition)
def fn(x, w, b):
    return torch.add(torch.matmul(x, w), b)

# Optionally compile it
fn = torch.compile(fn) if compile_fn else fn

# Annotate the function
def step():
    with torch.profiler.record_function("matmul_add"):
        return fn(x, w, b)

# Generate random inputs, weights, and biases
x = torch.randn(size, size, device=device, dtype=dtype)
w = torch.randn(size, size, device=device, dtype=dtype)
b = torch.randn(size, size, device=device, dtype=dtype)

# Optional warmup
if warmup_steps > 0:
    for _ in range(warmup_steps):
        step()
    torch.cuda.synchronize()

# Create the directory to store the trace
os.makedirs(trace_dir, exist_ok=True)

# Names for the trace files
compile_tag = "compile" if compile_fn else "eager"
tag = f"{size}_{dtype}_{warmup_steps}_{profile_steps}_{compile_tag}"
table_path = os.path.join(trace_dir, f"{tag}.txt")
trace_path = os.path.join(trace_dir, f"{tag}.json")

# Run profiling
schedule = torch.profiler.schedule(wait=1, warmup=1, active=3, repeat=1)
with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    schedule=schedule,
    record_shapes=False,
    profile_memory=False,
    with_stack=False,
) as prof:
    for _ in range(profile_steps):
        step()
        prof.step()
torch.cuda.synchronize()

# Save the traces
prof.export_chrome_trace(trace_path)
with open(table_path, "w") as f:
    f.write(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))

Print the table of results:

In [12]:
with open(table_path, 'r') as f:
    print(f.read())

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                             matmul_add         0.00%       0.000us         0.00%       0.000us       0.000us      26.016us       781.73%      26.016us      26.016us             1  
                                          ProfilerStep*        22.19%      85.655us        97.99%     378.237us     126.079us       0.000us         0.00%       3.328us       1.109us             3  
         